# NLP Auditor - Data Hygiene Layer
This notebook applies NLP to detect and correct mislabeled accidents.

In [ ]:
# Cell 1: Imports
import pandas as pd
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from nlp_utils import check_risk_keywords

In [ ]:
# Cell 2: Load Raw Data
df = pd.read_csv('../data/raw/synthetic_accident_data_v1.csv')
print(f"Loaded {len(df)} records.")

In [ ]:
# Cell 3: Apply NLP Audit
print("Running NLP Audit on descriptions...")
df['NLP_Risk_Flag'] = df['Incident_Description'].apply(check_risk_keywords)

In [ ]:
# Cell 4: The Correction Logic
# If Reported=0 BUT NLP_Flag=1 -> Change Severity to 1 (Major)
df['Corrected_Severity'] = df.apply(
    lambda row: 1 if (row['Reported_Severity'] == 0 and row['NLP_Risk_Flag'] == 1) 
    else row['Reported_Severity'], 
    axis=1
)

In [ ]:
# Cell 5: Visualize the Fix
misclassified_count = df[df['Reported_Severity'] != df['Corrected_Severity']].shape[0]
print(f"NLP Audit Complete!")
print(f"Found and Fixed {misclassified_count} mislabeled accidents.")

plt.figure(figsize=(6,4))
sns.countplot(x='Corrected_Severity', data=df)
plt.title('Severity Distribution AFTER Correction')
plt.show()

In [ ]:
# Cell 6: Save Processed Data
final_df = df.drop(columns=['Actual_Severity', 'Incident_Description'])
final_df.to_csv('../data/processed/training_data_cleaned.csv', index=False)
print("Cleaned data saved for modeling.")